# JIN AI — Phase 17: continued pretraining on Kaggle

**Read this first.** This notebook continues the Phase 14 checkpoint. It does not
start a new model, and it does not promise a capability gain.

Sessions are bounded by wall clock (`--session-max-minutes`); the global step budget
never changes between sessions. Every stage prints what it found before it acts, and
a missing input is a hard error rather than a silent fallback.

Two failure modes this notebook is written to make impossible:

* training from **random weights** because the parent or the resume checkpoint was
  not where the command looked;
* training on **old data** while believing it is corpus v7.

The repository is copied into `/kaggle/working` because `/kaggle/input` is
read-only and the pipeline writes data, checkpoints and logs beside the code.

## Stage B — configuration (edit this cell only)

Fill in the paths you actually attached. Nothing here guesses a dataset slug.

In [ ]:
import os, sys, json, shutil, subprocess

CONFIG = {
    # 1. The repository, attached as a Kaggle Dataset (or cloned).
    "repo_path": "/kaggle/input/jin-ai-repo/JIN-AI-LLM-from-zero-to-hero-ZHIEND",
    # 2. Phase 14 weights. Required — the run refuses to start without them.
    "checkpoint_path": "/kaggle/input/jin-phase14-checkpoint/best.pt",
    # 3. Wikipedia parquet, mounted as a Dataset directory.
    "wikipedia_path": "/kaggle/input/wikipedia-en-parquet",
    # 4. Licence facts copied from that dataset's card. Required by ingestion:
    #    a mounted folder name is not permission.
    "license_map_path": "/kaggle/working/license_map.json",
    # 5. Optional: a corpus prepared by an earlier session.
    "corpus_v7_path": None,
    # 6. Optional: the previous Phase 17 session's latest.pt.
    "phase17_ckpt_path": None,
    # 7. Frozen evaluation file (Phase 16 protocol). Optional per-run: if absent,
    #    Stage H reports that language modelling was NOT measured instead of
    #    inventing a number.
    "eval_jsonl": None,
    "out_dir": "/kaggle/working/phase17_output",
}
print(json.dumps(CONFIG, indent=2))

## Stage B2 — working copy and input checks

Copies the repo into `/kaggle/working/repo` (the mount is read-only), then checks
every required input before any GPU time is spent.

In [ ]:
WORK = "/kaggle/working/repo"
REPO = CONFIG["repo_path"]
OUT = CONFIG["out_dir"]
os.makedirs(OUT, exist_ok=True)

if os.path.isdir(REPO):
    shutil.copytree(REPO, WORK, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc"))
elif os.path.isdir(WORK):
    print("using existing working copy")
else:
    raise SystemExit(f"repo not found at {REPO} and no working copy at {WORK}")
os.chdir(WORK)

missing = [k for k in ("checkpoint_path",) if not os.path.exists(CONFIG[k])]
if missing:
    raise SystemExit("missing required input(s): " + ", ".join(
        f"{k}={CONFIG[k]}" for k in missing))
print("checkpoint:", CONFIG["checkpoint_path"],
      round(os.path.getsize(CONFIG["checkpoint_path"]) / 1e9, 2), "GB")
print("working tree:", WORK)

## Stage A — environment preflight

Records the accelerator you actually got (it varies), VRAM, CUDA, torch, CPU, RAM,
disk, the repo HEAD and the checkpoint SHA-256. `host_kind` distinguishes a real
Kaggle kernel from a local machine.

In [ ]:
!nvidia-smi || echo 'no nvidia-smi'
subprocess.run([sys.executable, "scripts/kaggle_preflight.py",
                "--checkpoint", CONFIG["checkpoint_path"],
                "--out", f"{OUT}/kaggle_preflight.json"], check=True)
preflight = json.load(open(f"{OUT}/kaggle_preflight.json"))
assert preflight["host"]["host_kind"] == "kaggle", (
    "this is not a Kaggle kernel; stop and do not report Kaggle numbers")
print("visible GPUs:", preflight["torch"]["device_count"],
      "| the trainer uses device 0 only")

## Stage C — corpus preflight (sample first, never `--all`)

`--input-root` makes the mounted Dataset visible to ingestion. `--source` selects
one reviewed source id; the licence comes from the dataset card via `--license-map`,
and ingestion refuses a source whose licence is not recorded.

In [ ]:
!python scripts/ingest_corpus.py --input-root {CONFIG['wikipedia_path']} --list

In [ ]:
# Fill these from the listing above and from the upstream dataset card.
WIKI_SOURCE = ""          # e.g. "wikipedia_en_20231101"
WIKI_DATASET_DIR = ""     # the mounted folder name, e.g. "wikipedia-en-20231101"

assert WIKI_SOURCE and WIKI_DATASET_DIR, "set WIKI_SOURCE and WIKI_DATASET_DIR first"
licence_map = {WIKI_DATASET_DIR: {
    "license": "",           # exact licence string from the dataset card
    "source_url": "",        # dataset card URL
    "text_columns": ["text"],
    "language": "en",
}}
assert licence_map[WIKI_DATASET_DIR]["license"] and licence_map[WIKI_DATASET_DIR]["source_url"], (
    "licence and source URL are required; a folder name is not permission")
json.dump(licence_map, open(CONFIG["license_map_path"], "w"), indent=2)
print(json.dumps(licence_map, indent=2))

In [ ]:
# Bounded sample: 10k documents is enough to project disk, RAM and tokens.
!python scripts/ingest_corpus.py --input-root {CONFIG['wikipedia_path']} \
    --source {WIKI_SOURCE} --license-map {CONFIG['license_map_path']} \
    --max-docs-per-source 10000 --out-root {OUT}/curated_v7_sample
!du -sh {OUT}/curated_v7_sample
print('Project full size from this sample before a 10 GB ingest.')

## Stage D — corpus v7 (only if Stage C projections fit)

Run one command at a time and read the audit before continuing.

In [ ]:
# 1) Full ingest into the working copy
# !python scripts/ingest_corpus.py --input-root {CONFIG['wikipedia_path']} \
#     --source {WIKI_SOURCE} --license-map {CONFIG['license_map_path']} \
#     --max-docs-per-source 1000000 --out-root data/curated_v7

# 2) Verify: dedup, splits, licence, PII. Read reports/*.json, not just exit code.
# !python scripts/verify_corpus.py --in-root data/curated_v7 --out-root data/curated_v7 \
#     --pii-policy exclude --license-policy enforce

# 3) Tokenise with an explicit mixture (weights are targets, not guarantees)
# !python scripts/build_tokenized_corpus.py --splits data/curated_v7/splits \
#     --out data/gpt2_corpus_v7 --max-train-tokens 1000000000 \
#     --mix {WIKI_SOURCE}:0.65,cosmopedia_v2_sample:0.25,prepared_v4:0.08,project_seed:0.02
print('Stage D pending: run the three commands above, in order.')

In [ ]:
# Training data selection. Stage E and G read TRAIN_DATA and nothing else, so the
# corpus cannot silently stay on an older shard.
DEFAULT_DATA = "data/gpt2_corpus_v7"
if os.path.isdir(DEFAULT_DATA) and os.path.exists(f"{DEFAULT_DATA}/manifest.json"):
    TRAIN_DATA = DEFAULT_DATA
else:
    TRAIN_DATA = ""
    print(f"{DEFAULT_DATA} not built yet; finish Stage D before training.")
print("TRAIN_DATA =", TRAIN_DATA or "(unset)")

## Stage E — smoke test from the Phase 14 weights

The first validation loss must be near the parent's (~4.7), not near
ln(50,257) ≈ 10.8, which is what random initialization scores. A loss near 10.8
means the weights were never transferred: stop and fix the input.

In [ ]:
assert TRAIN_DATA, "build corpus v7 in Stage D first (no silent fallback to old data)"
!python train_gpt2.py --preset gpt2_small_reference --data-dir {TRAIN_DATA} \
    --init-from {CONFIG['checkpoint_path']} --continue-pretrain \
    --out-dir {OUT}/smoke --seq-len 1024 --batch-size 4 --grad-accum 4 \
    --max-steps 20 --warmup 2 --lr 1e-4 --eval-interval 10 --eval-iters 4 \
    --save-interval 10 --seed 17

In [ ]:
m = json.load(open(f"{OUT}/smoke/latest.pt.manifest.json"))
assert m["init_mode"] == "continued_pretrain", m["init_mode"]
assert m["parent_sha256"], "parent SHA missing"
assert m["parent_sha256"] == preflight["checkpoint"]["sha256"], "parent is not the attached checkpoint"
print(json.dumps({k: m[k] for k in ("iteration", "tokens_seen", "init_mode",
                                    "parent_sha256", "additional_tokens_this_run")}, indent=2))

## Stage F — learning-rate pilot

Same parent, corpus, seed and geometry; three candidates; choose on stability and
clean validation, never on the lowest training loss.

In [ ]:
for lr in ("3e-5", "1e-4", "3e-4"):
    print(f"--- lr {lr} ---")
    !python train_gpt2.py --preset gpt2_small_reference --data-dir {TRAIN_DATA} \
        --init-from {CONFIG['checkpoint_path']} --continue-pretrain \
        --out-dir {OUT}/lr_pilot_{lr} --seq-len 1024 --batch-size 4 --grad-accum 8 \
        --max-steps 300 --warmup 20 --lr {lr} --eval-interval 100 --eval-iters 4 \
        --save-interval 100 --session-max-minutes 25 --seed 17

## Stage G — continued pretraining, one session

Fixed global budget, chosen before the first session and never changed afterwards:
4 × 8 × 1024 = 32,768 tokens/step, so 100 M additional tokens is 3,052 steps.

In [ ]:
TOKENS_PER_STEP = 4 * 8 * 1024
TARGET_ADDITIONAL_TOKENS = 100_000_000
TOTAL_STEPS = TARGET_ADDITIONAL_TOKENS // TOKENS_PER_STEP
print(TOTAL_STEPS, "steps x", TOKENS_PER_STEP, "tokens =", TOTAL_STEPS * TOKENS_PER_STEP)

!python train_gpt2.py --preset gpt2_small_reference --data-dir {TRAIN_DATA} \
    --init-from {CONFIG['checkpoint_path']} --continue-pretrain \
    --out-dir {OUT}/run_100m --seq-len 1024 --batch-size 4 --grad-accum 8 \
    --max-steps {TOTAL_STEPS} --warmup 100 --lr 1e-4 --eval-interval 250 \
    --eval-iters 8 --save-interval 250 --session-max-minutes 500 --seed 17

## Stage G2 — next sessions: resume the same experiment

`--resume-from` alone is enough; `--resume` is only needed for the default
`<out-dir>/latest.pt`. If the checkpoint is missing the trainer now **stops**
instead of starting over from scratch, and `--max-steps` must equal the first
session's value or the fingerprint check refuses the run.

In [ ]:
PREV = CONFIG["phase17_ckpt_path"]  # e.g. /kaggle/input/jin-phase17-s1/latest.pt
assert PREV and os.path.exists(PREV), "attach the previous session's latest.pt"
!python train_gpt2.py --preset gpt2_small_reference --data-dir {TRAIN_DATA} \
    --resume-from {PREV} --out-dir {OUT}/run_100m \
    --seq-len 1024 --batch-size 4 --grad-accum 8 \
    --max-steps {TOTAL_STEPS} --warmup 100 --lr 1e-4 --eval-interval 250 \
    --eval-iters 8 --save-interval 250 --session-max-minutes 500 --seed 17

## Stage H — evaluation and export

Language modelling uses the frozen Phase 16 protocol, and only when the frozen
evaluation file is attached; otherwise the result is recorded as NOT MEASURED.
Capability is measured with the frozen 73-task benchmark. Lower loss is not a
capability claim.

In [ ]:
CKPT = f"{OUT}/run_100m/best.pt"
assert os.path.exists(CKPT), "no best.pt yet"

if CONFIG["eval_jsonl"] and os.path.exists(CONFIG["eval_jsonl"]):
    !python scripts/eval_phase16.py --checkpoint {CKPT} \
        --val-jsonl {CONFIG['eval_jsonl']} --seq-len 1024 \
        --out {OUT}/benchmark_results/eval_clean.json
else:
    print("LANGUAGE MODELLING: NOT MEASURED (frozen eval file not attached)")

!python scripts/phase16_task_benchmark.py --checkpoint {CKPT} \
    --out {OUT}/benchmark_results/task_benchmark.json

In [ ]:
import hashlib
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for c in iter(lambda: fh.read(1 << 22), b''):
            h.update(c)
    return h.hexdigest()

man = json.load(open(f"{OUT}/run_100m/latest.pt.manifest.json"))
actual = sha(f"{OUT}/run_100m/latest.pt")
assert actual == man["sha256"], "checkpoint hash does not match its manifest"
print(json.dumps({"sha256": actual, "iteration": man["iteration"],
                  "additional_tokens": man["additional_tokens_this_run"],
                  "parent_sha256": man["parent_sha256"],
                  "trained_with": man.get("training_args", {}).get("data_dir")}, indent=2))
print("\nNext session needs: latest.pt, latest.pt.manifest.json, session_log.json")
print("Use 'Save Version' to publish this output as a Dataset, then attach it next time.")

## Recovery

* Session died before a save → the previous `latest.pt` is intact (atomic writes); rerun Stage G2 with the same `TOTAL_STEPS`.
* "resume requested but no checkpoint at …" → the path is wrong; that message exists so the run cannot restart from scratch by accident.
* "Training configuration changed" / "Step budget changed" → the recipe differs; compare `experiment_config` in both manifests instead of forcing it.
* Smoke loss ≈ 10.8 → weights were not transferred; check `--init-from` and the parent SHA.
* `/kaggle/input` is read-only → run everything from `/kaggle/working/repo` (Stage B2).